In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import yfinance as yf
import datetime
import xgboost

from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import minimize, least_squares, newton, differential_evolution, dual_annealing
from scipy.special import ndtr

from sklearn.model_selection import train_test_split
from sklearn.metrics import explained_variance_score, mean_absolute_percentage_error, mean_squared_error, mean_squared_log_error
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.metrics import r2_score
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
from scipy.optimize import brentq
from scipy.stats.qmc import LatinHypercube

from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from joblib import Parallel, delayed
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from itertools import repeat
import random
import xgboost as xgb_lib
import gc
import optuna
import json as _json

In [2]:
# Load the created Stata files

path = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\data_output_final\data_output_2003_top3_qual.dta"
data = pd.read_stata(path)
data = data.drop(["index"], axis = 1)


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# GPU-compatible Heston pricer
def heston_charf(phi, S0, V0, rf, lambd, tau, kappa, theta, sigma, rho):
    phi = phi.to(torch.complex64)

    a = kappa * theta
    b = kappa + lambd

    d = torch.sqrt((rho * sigma * phi * 1j - b)**2
        + sigma**2 * (phi**2 + 1j * phi) + 0j)

    g = (b - rho * sigma * phi * 1j - d) / (b - rho * sigma * phi * 1j + d)

    exp_dt = torch.exp(-d * tau)

    C = (
        rf * phi * 1j * tau
        + a / sigma**2 * (
            (b - rho * sigma * phi * 1j - d) * tau
            - 2 * torch.log((1 - g * exp_dt) / (1 - g))
        )
    )

    D = (
        (b - rho * sigma * phi * 1j - d) / sigma**2
        * (1 - exp_dt) / (1 - g * exp_dt)
    )

    log_cf = C + D * V0 + 1j * phi * torch.log(S0)

    log_cf = torch.clamp(log_cf.real, -100.0, 50.0) + 1j * log_cf.imag
    return torch.exp(log_cf)


def integrand(phi, S0, K, V0, rf, lambd, tau, kappa, theta, sigma, rho):
    arg = (S0, V0, rf, lambd, tau, kappa, theta, sigma, rho)

    phi = phi.to(torch.complex64)

    num = (
        torch.exp(rf * tau) * heston_charf(phi - 1j, *arg)
        - K * heston_charf(phi, *arg)
    )

    den = 1j * phi * K**(1j * phi)

    return num / den




_GL_N = 64
_gl_nodes_np, _gl_weights_np = np.polynomial.laguerre.laggauss(_GL_N)

def heston_price_gpu(S0, K, V0, rf, lambd, tau, kappa, theta, sigma, rho, n=128):
    
    batch_n = S0.shape[0]

    device = torch.device('cuda')
    S0 = torch.as_tensor(S0, device=device, dtype=torch.float64)
    K = torch.as_tensor(K, device=device, dtype=torch.float64)
    V0 = torch.as_tensor(V0, device=device, dtype=torch.float64)
    rf = torch.as_tensor(rf, device=device, dtype=torch.float64)
    lambd = torch.as_tensor(lambd, device=device, dtype=torch.float64)
    tau = torch.as_tensor(tau, device=device, dtype=torch.float64)
    kappa = torch.as_tensor(kappa, device=device, dtype=torch.float64)
    theta = torch.as_tensor(theta, device=device, dtype=torch.float64)
    sigma = torch.as_tensor(sigma, device=device, dtype=torch.float64)
    rho = torch.as_tensor(rho, device=device, dtype=torch.float64)

    args = (S0, K, V0, rf, lambd, tau, kappa, theta, sigma, rho)


    gl_x = torch.as_tensor(_gl_nodes_np, device=device, dtype=torch.float64)
    gl_w = torch.as_tensor(_gl_weights_np, device=device, dtype=torch.float64)
    gl_w_adj = gl_w * torch.exp(gl_x)

    phi_c = gl_x.to(torch.complex64)
    phi_grid = phi_c.unsqueeze(0).expand(batch_n, -1)

    args_batch = []
    for i in args:
        if i.ndim == 1:
            args_batch.append(i.unsqueeze(-1))
        else:
            args_batch.append(i.reshape(1,1).expand(batch_n, -1))

    vals = integrand(phi_grid, *args_batch)
    integral = torch.sum(vals * gl_w_adj.unsqueeze(0), dim=1)
    real_integral = torch.real(integral)
    call = (S0 - K * torch.exp(-rf * tau)) / 2.0 + real_integral / torch.pi

    return torch.maximum(call, torch.tensor(0.0, device=device))




def put_heston_gpu(call_price, S0, K, rf, tau):
    device = torch.device('cuda')
    S0 = torch.as_tensor(S0, device=device, dtype=torch.float64)
    K = torch.as_tensor(K, device=device, dtype=torch.float64)
    rf = torch.as_tensor(rf, device=device, dtype=torch.float64)
    tau = torch.as_tensor(tau, device=device, dtype=torch.float64)
    put = torch.exp(-rf*tau)*K + call_price - S0

    return torch.maximum(put, torch.tensor(0.0, device=device))


def heston_avg_var(V0, kappa, theta, tau):
    device = (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
    V0    = torch.as_tensor(V0.to_numpy(),    device=device, dtype=torch.float64)
    kappa = torch.as_tensor(kappa.to_numpy(), device=device, dtype=torch.float64)
    theta = torch.as_tensor(theta.to_numpy(), device=device, dtype=torch.float64)
    tau   = torch.as_tensor(tau.to_numpy(),   device=device, dtype=torch.float64)

    safe_kappa = kappa.clamp(min=1e-8)
    avg_var = theta + (V0 - theta) * (1.0 - torch.exp(-safe_kappa * tau)) / (safe_kappa * tau)
    return torch.sqrt(avg_var.clamp(min=0.0))





def SABR_vol(alpha,beta,rho,nu,F,K,tau):
    device = torch.device('cuda')
    eps = 1e-8
    alpha = torch.as_tensor(alpha, device=device, dtype=torch.float64)
    beta = torch.as_tensor(beta, device=device, dtype=torch.float64)
    rho = torch.as_tensor(rho, device=device, dtype=torch.float64)
    nu = torch.as_tensor(nu, device=device, dtype=torch.float64)
    F = torch.as_tensor(F, device=device, dtype=torch.float64)
    K = torch.as_tensor(K, device=device, dtype=torch.float64)
    tau = torch.as_tensor(tau, device=device, dtype=torch.float64)
    F = torch.clamp(torch.as_tensor(F, device=device, dtype=torch.float64), min=eps)
    K = torch.clamp(torch.as_tensor(K, device=device, dtype=torch.float64), min=eps)

    condition = torch.isclose(F, K, rtol=1e-5)

    V = (F*K)**((1-beta)/2.)
    logFK = torch.log(torch.clamp(F/K, min=eps))
    A = 1 + ( ((1-beta)**2*alpha**2)/(24.*(V**2)) + (alpha*beta*nu*rho)/(4.*V) + ((nu**2)*(2-3*(rho**2))/24.) ) * tau
    
    VOL_eq = (alpha/V)*A

    z = (nu/alpha)*V*logFK
    x = torch.log((torch.sqrt(1-2*rho*z+z**2) + z - rho)/(1-rho))
    B = 1 + (1/24.)*(((1-beta)*logFK)**2) + (1/1920.)*(((1-beta)*logFK)**4)
    
    VOL_ineq = (nu*logFK*A)/(x*B)

    VOL = torch.where(condition, VOL_eq, VOL_ineq)

    return (VOL)


def BS_pricer(S0, K, rf, impl_vol, tau, corp):
    
    batch_n = S0.shape[0]

    device = torch.device('cuda')
    S0 = torch.as_tensor(S0, device=device, dtype=torch.float64)
    K = torch.as_tensor(K, device=device, dtype=torch.float64)
    rf = torch.as_tensor(rf, device=device, dtype=torch.float64)
    impl_vol = torch.as_tensor(impl_vol, device=device, dtype=torch.float64)
    tau = torch.as_tensor(tau, device=device, dtype=torch.float64)
    corp = torch.as_tensor(corp, device=device, dtype=torch.float64)
    
    if not torch.isfinite(impl_vol).all():
        print("impl_vol:", impl_vol)
        print(impl_vol[~torch.isfinite(impl_vol)])
    d1 = 1/(impl_vol*torch.sqrt(tau))*(torch.log(S0/K) + (rf + (impl_vol**2)/2)*tau)
    d2 = d1 - impl_vol*torch.sqrt(tau)
    BS_price = corp*torch.distributions.Normal(0, 1).cdf(corp*d1)*S0 - corp*torch.distributions.Normal(0, 1).cdf(corp*d2)*K*torch.exp(-rf*tau)
    return(BS_price)


def BS_impl_vol(S0, K, rf, P, tau, corp):
    def obj_funct(impl_vol_guess):
        return((P - BS_pricer(S0, K, rf, impl_vol_guess, tau, corp))**2)
    
    impl_vol_guess = torch.tensor([0.03], requires_grad=True, device='cuda')


    optimizer = optim.Adam([impl_vol_guess], lr=0.01)

    num_iterations = 1000
    for i in range(num_iterations):
        optimizer.zero_grad()
        loss = obj_funct(impl_vol_guess)
        loss.backward()
        optimizer.step()
    
    for i in range(num_iterations):
        y = obj_funct(impl_vol_guess)
        grad = torch.autograd.grad(y, impl_vol_guess, create_graph=True)[0]
        hess = torch.autograd.grad(grad, impl_vol_guess, create_graph=True)[0]
        impl_vol_guess = impl_vol_guess - grad/hess


    impl_var_optimized = impl_vol_guess.item()
    impl_vol_optimized = torch.sqrt(impl_vol_guess).item()

    print(f"Optimized Variance: {impl_var_optimized}")
    print(f"Instantaneous Volatility: {impl_vol_optimized}")

    return(impl_vol_optimized)

In [ ]:
# GPU Implied Vol estimator (Newton Rhapson method) with batching and parallelization

def bs_price_torch(S, K, T, r, cp, sigma):
    eps = 1e-12
    sigma = torch.clamp(sigma, min=eps)
    T = torch.clamp(T, min=eps)

    d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * torch.sqrt(T))
    d2 = d1 - sigma * torch.sqrt(T)

    normal = torch.distributions.Normal(0.0, 1.0)
    return cp * (S * normal.cdf(cp * d1) - K * torch.exp(-r * T) * normal.cdf(cp * d2))

def bs_vega_torch(S, K, T, r, sigma):
    eps = 1e-12
    sigma = torch.clamp(sigma, min=eps)
    T = torch.clamp(T, min=eps)

    d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * torch.sqrt(T))
    normal = torch.distributions.Normal(0.0, 1.0)
    return S * normal.log_prob(d1).exp() * torch.sqrt(T)


def implied_vol_newton_gpu_batched(
    dfs: list[pd.DataFrame],
    price_cols: list[str],
    batch_size: int = 10_000,
    initial_guess: float = 0.2,
    tol: float = 1e-10,
    max_iter: int = 100,
) -> list[torch.Tensor]:
    dtype  = torch.float64
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n      = len(dfs[0])
    N      = len(dfs)

    S_np     = [df["stock_price"].values  for df in dfs]
    K_np     = [df["strike_price"].values for df in dfs]
    T_np     = [df["time_to_mat"].values  for df in dfs]
    r_np     = [df["rf"].values           for df in dfs]
    cp_np    = [df["cp_flag_one"].values  for df in dfs]
    price_np = [df[col].values for df, col in zip(dfs, price_cols)]

    results = [torch.empty(n, dtype=dtype) for _ in range(N)]

    for batch_start in range(0, n, batch_size):
        batch_end = min(batch_start + batch_size, n)
        sl = slice(batch_start, batch_end)

        def to_gpu(arr):
            return torch.as_tensor(arr[sl], dtype=dtype, device=device)

        S  = torch.stack([to_gpu(s)  for s  in S_np],  dim=0)
        K  = torch.stack([to_gpu(k)  for k  in K_np],  dim=0)
        T  = torch.stack([to_gpu(t)  for t  in T_np],  dim=0)
        r  = torch.stack([to_gpu(rv) for rv in r_np],  dim=0)
        cp = torch.stack([to_gpu(c)  for c  in cp_np], dim=0)

        prices = torch.stack(
            [torch.as_tensor(p[sl], dtype=dtype, device=device) for p in price_np],
            dim=0,
        )

        sigma = torch.full((N, batch_end - batch_start), initial_guess,
                           dtype=dtype, device=device)

        for _ in range(max_iter):
            price_est = bs_price_torch(S, K, T, r, cp, sigma)
            vega      = bs_vega_torch(S, K, T, r, sigma)

            diff      = price_est - prices
            converged = torch.abs(diff) < tol

            if torch.all(converged):
                break

            vega  = torch.clamp(vega, min=1e-12)
            sigma = sigma - diff / vega
            sigma = torch.clamp(sigma, min=1e-8, max=5.0)

        for i in range(N):
            results[i][sl] = sigma[i].cpu()

        del S, K, T, r, cp, prices, sigma, price_est, vega, diff, converged
        torch.cuda.empty_cache()

    return results


def bs_price_from_iv_gpu_batched(
    dfs: list[pd.DataFrame],
    iv_cols: list[str],
    batch_size: int = 10_000,
) -> list[torch.Tensor]:
    dtype  = torch.float64
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n      = len(dfs[0])
    N      = len(dfs)

    S_np  = [df["stock_price"].values  for df in dfs]
    K_np  = [df["strike_price"].values for df in dfs]
    T_np  = [df["time_to_mat"].values  for df in dfs]
    r_np  = [df["rf"].values           for df in dfs]
    cp_np = [df["cp_flag_one"].values  for df in dfs]
    iv_np = [df[col].values for df, col in zip(dfs, iv_cols)]

    results = [torch.empty(n, dtype=dtype) for _ in range(N)]

    for batch_start in range(0, n, batch_size):
        batch_end = min(batch_start + batch_size, n)
        sl = slice(batch_start, batch_end)

        def to_gpu(arr):
            return torch.as_tensor(arr[sl], dtype=dtype, device=device)

        S  = torch.stack([to_gpu(s)  for s  in S_np],  dim=0)
        K  = torch.stack([to_gpu(k)  for k  in K_np],  dim=0)
        T  = torch.stack([to_gpu(t)  for t  in T_np],  dim=0)
        r  = torch.stack([to_gpu(rv) for rv in r_np],  dim=0)
        cp = torch.stack([to_gpu(c)  for c  in cp_np], dim=0)

        sigmas = torch.stack(
            [torch.as_tensor(iv[sl], dtype=dtype, device=device) for iv in iv_np],
            dim=0,
        )

        prices = bs_price_torch(S, K, T, r, cp, sigmas)

        for i in range(N):
            results[i][sl] = prices[i].cpu()

        del S, K, T, r, cp, sigmas, prices
        torch.cuda.empty_cache()

    return results

In [ ]:
# Optuna: pre-computation helpers & objective functions

optuna.logging.set_verbosity(optuna.logging.WARNING)

_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def _compute_sv_batch(df, params_np, model_type, batch_size=2048):
    device = _device
    N = len(df); out = np.zeros(N, dtype=np.float64)
    for s in range(0, N, batch_size):
        e = min(s + batch_size, N)
        def t(arr): return torch.tensor(arr, dtype=torch.float64, device=device)
        if model_type == "heston":
            S0  = t(df["stock_price"].values[s:e]); K  = t(df["strike_price"].values[s:e])
            tau = t(df["time_to_mat"].values[s:e]); rf = t(df["rf"].values[s:e])
            lam = t(df["VRP"].values[s:e]);         cp = t(df["cp_flag_one"].values[s:e])
            V0  = t(params_np[s:e,0]); kap = torch.tensor(2.0, dtype=torch.float64, device=device)
            tht = t(params_np[s:e,1]); sig = t(params_np[s:e,2]); rho = t(params_np[s:e,3])
            call  = heston_price_gpu(S0, K, V0, rf, lam, tau, kap, tht, sig, rho)
            price = torch.where(cp > 0, call, put_heston_gpu(call, S0, K, rf, tau))
            out[s:e] = price.cpu().numpy()
        else:
            F   = t(df["forward_price"].values[s:e]); K  = t(df["strike_price"].values[s:e])
            tau = t(df["time_to_mat"].values[s:e])
            S0  = t(df["stock_price"].values[s:e]); rf = t(df["rf"].values[s:e])
            cp  = t(df["cp_flag_one"].values[s:e])
            alp = t(params_np[s:e,0]); bet = torch.ones(e-s, dtype=torch.float64, device=device)
            rho = t(params_np[s:e,1]); nu  = t(params_np[s:e,2])
            sabr_ivol = SABR_vol(alp, bet, rho, nu, F, K, tau)
            out[s:e] = BS_pricer(S0, K, rf, sabr_ivol, tau, cp).cpu().numpy()
    return out


def precompute_two_step(data, model_type, seed=42):
    set_seed(seed)
    if model_type == "heston":
        param_names  = ["V0", "theta", "sigma", "rho"]
        param_bounds = [(1e-3, 0.30), (1e-3, 0.30), (1e-2, 1.0), (-0.999, 0.0)]
        mkt_cols     = ["stock_price", "strike_price", "time_to_mat", "rf", "VRP", "cp_flag_one"]
    else:
        param_names  = ["alphaS", "rhoS", "nuS"]
        param_bounds = [(1e-3, 10.0), (-0.999, -0.001), (1e-3, 10.0)]
        mkt_cols     = ["forward_price", "stock_price", "strike_price", "time_to_mat", "rf", "cp_flag_one"]

    n_params = len(param_names)
    no_scale_cols = {"cp_flag_one"}

    df = data.copy().reset_index(drop=True)
    if "cp_flag_one" not in df.columns:
        df["cp_flag_one"] = df["cp_flag"].apply(lambda x: 1.0 if x == "C" else -1.0)

    dates  = df["date"].sort_values().unique()
    cutoff = dates[int(len(dates) * 0.7)]
    train_all = df[df["date"] < cutoff].copy().reset_index(drop=True)
    split = len(train_all) // 2
    set_a = train_all.iloc[:split].copy().reset_index(drop=True)

    lhs_unit   = LatinHypercube(d=n_params, seed=seed).random(n=len(set_a))
    params_lhs = np.array([lb + (ub-lb)*lhs_unit[:,j]
                           for j,(lb,ub) in enumerate(param_bounds)]).T
    print(f"  Computing stoch-vol prices ({model_type}, {len(set_a):,} rows)...")
    sv_a  = _compute_sv_batch(set_a, params_lhs, model_type)
    valid = np.isfinite(sv_a) & (sv_a > 0)
    print(f"  Valid: {valid.sum():,} / {len(sv_a):,}")

    set_a_v  = set_a[valid].reset_index(drop=True)
    params_v = params_lhs[valid]
    sv_v     = sv_a[valid]

    X_mkt = set_a_v[mkt_cols].values.astype(np.float32)
    X_all = np.concatenate([X_mkt, params_v.astype(np.float32)], axis=1)
    y_all = np.log(sv_v).astype(np.float32)
    n_feat = X_all.shape[1]

    n_tr = int(len(y_all) * 0.85)
    X_tr, X_val = X_all[:n_tr], X_all[n_tr:]
    y_tr, y_val = y_all[:n_tr], y_all[n_tr:]

    scale_mask = np.array([c not in no_scale_cols for c in mkt_cols] + [True]*n_params)
    scaler_x = StandardScaler().fit(X_tr[:, scale_mask])
    def _do_scale(X):
        Xs = X.copy()
        Xs[:, scale_mask] = scaler_x.transform(X[:, scale_mask])
        return Xs.astype(np.float32)

    X_tr_sc  = _do_scale(X_tr)
    X_val_sc = _do_scale(X_val)

    print(f"  n_feat={n_feat}, train={len(X_tr):,}, val={len(X_val):,}")
    return dict(
        X_tr_sc=X_tr_sc, X_val_sc=X_val_sc,
        X_tr_raw=X_tr,   X_val_raw=X_val,
        y_tr=y_tr, y_val=y_val,
        n_feat=n_feat, n_out=1,
    )


def precompute_one_step(data, model_type, seed=42):
    set_seed(seed)
    if model_type == "heston":
        param_names  = ["V0", "theta", "sigma", "rho"]
        param_bounds = [(1e-3, 0.30), (1e-3, 0.30), (1e-2, 1.0), (-0.999, 0.0)]
        attr_cols    = ["stock_price", "strike_price", "time_to_mat", "rf", "VRP", "cp_flag_one"]
        sv_col       = "price"
    else:
        param_names  = ["alphaS", "rhoS", "nuS"]
        param_bounds = [(1e-3, 10.0), (-0.999, -0.001), (1e-3, 10.0)]
        attr_cols    = ["forward_price", "stock_price", "strike_price", "time_to_mat", "rf", "cp_flag_one"]
        sv_col       = "price"

    n_params = len(param_names)
    no_scale_cols = {"cp_flag_one"}

    df = data.copy().reset_index(drop=True)
    if "cp_flag_one" not in df.columns:
        df["cp_flag_one"] = df["cp_flag"].apply(lambda x: 1.0 if x == "C" else -1.0)

    dates  = df["date"].sort_values().unique()
    cutoff = dates[int(len(dates) * 0.7)]
    train = df[df["date"] < cutoff].copy().reset_index(drop=True)

    lhs_unit   = LatinHypercube(d=n_params, seed=seed).random(n=len(train))
    params_lhs = np.array([lb + (ub-lb)*lhs_unit[:,j]
                           for j,(lb,ub) in enumerate(param_bounds)]).T
    print(f"  Computing stoch-vol prices ({model_type}, {len(train):,} rows)...")
    sv_train = _compute_sv_batch(train, params_lhs, model_type)
    valid    = np.isfinite(sv_train)
    print(f"  Valid: {valid.sum():,} / {len(sv_train):,}")

    train_v  = train[valid].reset_index(drop=True)
    params_v = params_lhs[valid]
    sv_v     = sv_train[valid]

    X_all = np.concatenate([
        train_v[attr_cols].values.astype(np.float32),
        sv_v.reshape(-1,1).astype(np.float32),
    ], axis=1)
    Y_all = params_v.astype(np.float32)
    n_feat = X_all.shape[1]

    n_tr = int(len(Y_all) * 0.85)
    X_tr, X_val = X_all[:n_tr], X_all[n_tr:]
    Y_tr, Y_val = Y_all[:n_tr], Y_all[n_tr:]

    scaler_y = StandardScaler().fit(Y_tr)
    Y_tr_sc = scaler_y.transform(Y_tr).astype(np.float32)
    Y_val_sc = scaler_y.transform(Y_val).astype(np.float32)

    all_x_cols = attr_cols + [sv_col]
    scale_mask = np.array([c not in no_scale_cols for c in all_x_cols])
    scaler_x = StandardScaler().fit(X_tr[:, scale_mask])
    def _do_scale(X):
        Xs = X.copy()
        Xs[:, scale_mask] = scaler_x.transform(X[:, scale_mask])
        return Xs.astype(np.float32)

    X_tr_sc  = _do_scale(X_tr)
    X_val_sc = _do_scale(X_val)

    print(f"  n_feat={n_feat}, n_params={n_params}, train={len(X_tr):,}, val={len(X_val):,}")
    return dict(
        X_tr_sc=X_tr_sc, X_val_sc=X_val_sc,
        X_tr_raw=X_tr,   X_val_raw=X_val,
        Y_tr_sc=Y_tr_sc, Y_val_sc=Y_val_sc,
        n_feat=n_feat, n_out=n_params,
    )



def _objective_xgb(trial, X_tr, y_tr, X_val, y_val, multi_output=False):
    p = dict(
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3,  log=True),
        max_depth     = trial.suggest_int(  'max_depth',     3,    12),
        n_estimators  = trial.suggest_int(  'n_estimators',  50,   500),
        subsample     = trial.suggest_float('subsample',     0.5,  1.0),
        reg_lambda    = trial.suggest_float('reg_lambda',    0.01, 10.0, log=True),
        gamma         = trial.suggest_float('gamma',         0.0,  5.0),
        eval_metric='rmse', tree_method='hist', device='cuda',
    )
    if multi_output:
        m = MultiOutputRegressor(
            xgb_lib.XGBRegressor(**p), n_jobs=1,
        )
        m.fit(X_tr, y_tr)
        result = mean_squared_error(y_val, m.predict(X_val))
    else:
        m = xgb_lib.XGBRegressor(**p)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        result = mean_squared_error(y_val, m.predict(X_val))
    del m
    return result


def _objective_ann(trial, Xtr_t, Ytr_t, Xv_t, Yv_t, n_out=1):
    n_layers = trial.suggest_int(        'n_layers', 2,   4)
    n_units  = trial.suggest_categorical('n_units',  [32, 64, 128])
    lr       = trial.suggest_float(      'lr',       1e-4, 1e-2, log=True)
    dropout  = trial.suggest_float(      'dropout',  0.0,  0.4)
    device   = Xtr_t.device
    try:
        layers = [nn.Linear(Xtr_t.shape[1], n_units), nn.GELU()]
        for _ in range(n_layers - 1):
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            layers += [nn.Linear(n_units, n_units), nn.GELU()]
        layers.append(nn.Linear(n_units, n_out))
        model = nn.Sequential(*layers).to(device).double()
        opt   = torch.optim.Adam(model.parameters(), lr=lr)
        crit  = nn.MSELoss()

        best, patience, counter = float('inf'), 30, 0
        for epoch in range(200):
            model.train()
            opt.zero_grad()
            pred = model(Xtr_t)
            if n_out == 1: pred = pred.squeeze(-1)
            loss = crit(pred, Ytr_t)
            loss.backward()
            opt.step()
            model.eval()
            with torch.no_grad():
                vpred = model(Xv_t)
                if n_out == 1: vpred = vpred.squeeze(-1)
                vl = crit(vpred, Yv_t).item()
            trial.report(vl, epoch)
            if trial.should_prune():
                del model, opt
                torch.cuda.empty_cache()
                raise optuna.exceptions.TrialPruned()
            if vl < best:
                best, counter = vl, 0
            else:
                counter += 1
                if counter >= patience:
                    break
        del model, opt
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        return best
    except RuntimeError as e:
        if 'CUDA' in str(e):
            try: torch.cuda.synchronize()
            except RuntimeError: pass
            torch.cuda.empty_cache()
            return float('inf')
        raise


def _objective_lbfgs(trial, Xtr_t, Ytr_t, Xv_t, Yv_t, n_out=1):
    n_layers     = trial.suggest_int(        'n_layers',     2,  4)
    n_units      = trial.suggest_categorical('n_units',      [32, 64, 128])
    max_iter     = trial.suggest_int(        'max_iter',     20, 50)
    history_size = trial.suggest_int(        'history_size', 5,  15)
    device       = Xtr_t.device
    try:
        layers = [nn.Linear(Xtr_t.shape[1], n_units), nn.GELU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_units, n_units), nn.GELU()]
        layers.append(nn.Linear(n_units, n_out))
        model = nn.Sequential(*layers).to(device).double()
        opt   = torch.optim.LBFGS(model.parameters(), max_iter=max_iter,
                                   history_size=history_size, line_search_fn='strong_wolfe')
        crit  = nn.MSELoss()

        best, patience, counter = float('inf'), 10, 0
        for epoch in range(30):
            model.train()
            def closure():
                opt.zero_grad()
                pred = model(Xtr_t)
                if n_out == 1: pred = pred.squeeze(-1)
                loss = crit(pred, Ytr_t)
                if torch.isnan(loss) or torch.isinf(loss):
                    return loss
                loss.backward()
                return loss
            opt.step(closure)
            model.eval()
            with torch.no_grad():
                vpred = model(Xv_t)
                if n_out == 1: vpred = vpred.squeeze(-1)
                vl = crit(vpred, Yv_t).item()
            if np.isnan(vl) or np.isinf(vl):
                break
            if vl < best:
                best, counter = vl, 0
            else:
                counter += 1
                if counter >= patience:
                    break
        del model, opt
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        return best
    except RuntimeError as e:
        if 'CUDA' in str(e):
            try: torch.cuda.synchronize()
            except RuntimeError: pass
            torch.cuda.empty_cache()
            return float('inf')
        raise

print("Optuna helpers loaded.")

Optuna helpers loaded.


c:\Users\chril\torch_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Run all 12 Optuna studies

OPTUNA_N_TRIALS_XGB  = 50
OPTUNA_N_TRIALS_ANN  = 20
SAVE_PATH = 'C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\optuna_best_params_2003.json'
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=20)
optuna_results = {}

set_seed(42)

for approach in ["two_step", "one_step"]:
    for model_type in ["heston", "sabr"]:
        print(f"\n{'='*70}")
        print(f"  {approach.upper()} | {model_type.upper()} — pre-computing data...")
        print(f"{'='*70}")

        if approach == "two_step":
            pc = precompute_two_step(data, model_type, seed=42)
            multi_output = False
        else:
            pc = precompute_one_step(data, model_type, seed=42)
            multi_output = True

        n_out = pc['n_out']

        if approach == "two_step":
            _Xtr_t = torch.tensor(pc['X_tr_sc'], dtype=torch.float64, device=_device)
            _Ytr_t = torch.tensor(pc['y_tr'],    dtype=torch.float64, device=_device)
            _Xv_t  = torch.tensor(pc['X_val_sc'], dtype=torch.float64, device=_device)
            _Yv_t  = torch.tensor(pc['y_val'],    dtype=torch.float64, device=_device)
        else:
            _Xtr_t = torch.tensor(pc['X_tr_sc'],  dtype=torch.float64, device=_device)
            _Ytr_t = torch.tensor(pc['Y_tr_sc'],  dtype=torch.float64, device=_device)
            _Xv_t  = torch.tensor(pc['X_val_sc'], dtype=torch.float64, device=_device)
            _Yv_t  = torch.tensor(pc['Y_val_sc'], dtype=torch.float64, device=_device)

        key = f"{approach}_xgb_{model_type}"
        print(f"\n  XGB ({OPTUNA_N_TRIALS_XGB} trials) → {key}")
        if approach == "two_step":
            xgb_X_tr, xgb_y_tr = pc['X_tr_raw'], pc['y_tr']
            xgb_X_val, xgb_y_val = pc['X_val_raw'], pc['y_val']
        else:
            xgb_X_tr, xgb_y_tr = pc['X_tr_raw'], pc['Y_tr_sc']
            xgb_X_val, xgb_y_val = pc['X_val_raw'], pc['Y_val_sc']

        study = optuna.create_study(direction='minimize')
        study.optimize(
            lambda t: _objective_xgb(t, xgb_X_tr, xgb_y_tr, xgb_X_val, xgb_y_val,
                                     multi_output=multi_output),
            n_trials=OPTUNA_N_TRIALS_XGB, show_progress_bar=True,
        )
        optuna_results[key] = study.best_params
        print(f"    best val MSE: {study.best_value:.6f}  params: {study.best_params}")
        del study; gc.collect()
        torch.cuda.synchronize(); torch.cuda.empty_cache()


        key = f"{approach}_lbfgs_{model_type}"
        print(f"\n  ANN-LBFGS ({OPTUNA_N_TRIALS_ANN} trials) → {key}")
        study = optuna.create_study(direction='minimize')
        study.optimize(
            lambda t: _objective_lbfgs(t, _Xtr_t, _Ytr_t, _Xv_t, _Yv_t, n_out=n_out),
            n_trials=OPTUNA_N_TRIALS_ANN, show_progress_bar=True,
            catch=(RuntimeError,),
        )
        optuna_results[key] = study.best_params
        print(f"    best val MSE: {study.best_value:.6f}  params: {study.best_params}")
        del study; gc.collect()
        torch.cuda.synchronize(); torch.cuda.empty_cache()

        del _Xtr_t, _Ytr_t, _Xv_t, _Yv_t, pc
        gc.collect()
        torch.cuda.synchronize(); torch.cuda.empty_cache()

with open(SAVE_PATH, 'w') as f:
    _json.dump(optuna_results, f, indent=2)
print(f"\nSaved {len(optuna_results)} param sets to {SAVE_PATH}")
print(optuna_results)


  TWO_STEP | HESTON — pre-computing data...
  Computing stoch-vol prices (heston, 58,532 rows)...
  Valid: 57,700 / 58,532
  n_feat=10, train=49,045, val=8,655

  XGB (50 trials) → two_step_xgb_heston


  0%|          | 0/50 [00:00<?, ?it/s]c:\Users\chril\torch_gpu\Lib\site-packages\xgboost\core.py:751: UserWarning: [01:35:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Best trial: 40. Best value: 0.480832: 100%|██████████| 50/50 [02:30<00:00,  3.01s/it]


    best val MSE: 0.480832  params: {'learning_rate': 0.03510733639050293, 'max_depth': 11, 'n_estimators': 308, 'subsample': 0.5112582896550547, 'reg_lambda': 6.8979564951313534, 'gamma': 0.007779667869642751}

  ANN-LBFGS (20 trials) → two_step_lbfgs_heston


Best trial: 9. Best value: 0.0115352: 100%|██████████| 20/20 [56:57<00:00, 170.89s/it] 


    best val MSE: 0.011535  params: {'n_layers': 3, 'n_units': 128, 'max_iter': 40, 'history_size': 15}

  TWO_STEP | SABR — pre-computing data...
  Computing stoch-vol prices (sabr, 58,532 rows)...
  Valid: 45,085 / 58,532
  n_feat=9, train=38,322, val=6,763

  XGB (50 trials) → two_step_xgb_sabr


...

    best val MSE: 0.757228  params: {'n_layers': 3, 'n_units': 128, 'max_iter': 50, 'history_size': 11}

  ONE_STEP | SABR — pre-computing data...
  Computing stoch-vol prices (sabr, 117,065 rows)...
  Valid: 117,065 / 117,065
  n_feat=7, n_params=3, train=99,505, val=17,560

  XGB (50 trials) → one_step_xgb_sabr


Best trial: 42. Best value: 0.641021: 100%|██████████| 50/50 [03:56<00:00,  4.73s/it]


    best val MSE: 0.641021  params: {'learning_rate': 0.020464381773418736, 'max_depth': 12, 'n_estimators': 291, 'subsample': 0.9029270751913427, 'reg_lambda': 2.585993121087356, 'gamma': 4.54577045901569}

  ANN-LBFGS (20 trials) → one_step_lbfgs_sabr


Best trial: 13. Best value: 0.618333: 100%|██████████| 20/20 [1:31:58<00:00, 275.90s/it]


    best val MSE: 0.618333  params: {'n_layers': 3, 'n_units': 64, 'max_iter': 46, 'history_size': 13}

Saved 8 param sets to C:\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\optuna_best_params_2003.json
{'two_step_xgb_heston': {'learning_rate': 0.03510733639050293, 'max_depth': 11, 'n_estimators': 308, 'subsample': 0.5112582896550547, 'reg_lambda': 6.8979564951313534, 'gamma': 0.007779667869642751}, 'two_step_lbfgs_heston': {'n_layers': 3, 'n_units': 128, 'max_iter': 40, 'history_size': 15}, 'two_step_xgb_sabr': {'learning_rate': 0.19316082865988835, 'max_depth': 8, 'n_estimators': 247, 'subsample': 0.5528008598208944, 'reg_lambda': 5.60767068667638, 'gamma': 2.1636516129611}, 'two_step_lbfgs_sabr': {'n_layers': 4, 'n_units': 64, 'max_iter': 36, 'history_size': 9}, 'one_step_xgb_heston': {'learning_rate': 0.15481298363980872, 'max_depth': 7, 'n_estimators': 459, 'subsample': 0.9508777032363638, 'reg_lambda': 1.663283